In [25]:
import sys 
import os 
sys.path.append('/volume_4/research/seongbong/flywire/geosmin_project_version_update/code')
from analysis_of_simulation_result import * 
from fitting import *
from plot_of_experiment_data import std_err
import matplotlib.pyplot as plt


from tqdm import tqdm
from scipy.interpolate import RegularGridInterpolator
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import interp1d



# Load experimental data

In [2]:
from pathlib import Path
ROOT = Path("/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure4/PER_fitting_final/data/Experiment_data_final")

DATASETS = {
    "tarsal_all": {
        "path": ROOT / "tarsal_PER.xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
    "tarsal_TPN1": {
        "path": ROOT / "atGRN_Kir2.1.xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
    "tarsal_atGRN": {
        "path": ROOT / "Gr5a_Kir2.1.xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
    "labial_all": {
        "path": ROOT / "labial_PER.xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
    "labial_L1L2": {
        "path": ROOT / "Ir56b_Kir2.1(L1+L2).xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
    "labial_L3": {
        "path": ROOT / "Gr5a-Ir56b_Kir2.1(L3).xlsx",
        "sugar_sheet": "sugar",
        "geosmin_sheet": "sugar+geosmin",
    },
}

experiment_data = {
    name: load_per_pair(
        excel_path=info["path"],
        sugar_sheet=info["sugar_sheet"],
        geosmin_sheet=info["geosmin_sheet"],
    )
    for name, info in DATASETS.items()
}



# Load simulation data

In [3]:
from scipy.interpolate import RegularGridInterpolator
at_vals = np.arange(0, 201, 20)   # [0,20,...,200] → 11개
tp_vals = np.arange(0, 201, 20)   # 11개

Z_sim_tarsal = np.array(pd.read_parquet('data/simulation_data_final/tarsal/tarsal_sugar.parquet'))
Z_sim_geo_tarsal = np.array(pd.read_parquet('data/simulation_data_final/tarsal/tarsal_sugar_geosmin_170Hz.parquet'))


Z_sim_labial = np.array(pd.read_parquet('data/simulation_data_final/labellar/labial_sugar.parquet'))
Z_sim_geo_labial = np.array(pd.read_parquet('data/simulation_data_final/labellar/labial_sugar_geosmin_170Hz.parquet'))


interp_func_labial = make_interpolator(Z_sim_labial, at_vals)
interp_func_geo_labial = make_interpolator(Z_sim_geo_labial, at_vals)

interp_func_tarsal = make_interpolator(Z_sim_tarsal, at_vals)
interp_func_geo_tarsal = make_interpolator(Z_sim_geo_tarsal, at_vals)

# Taral Fitting 

In [31]:


for rate in list(range(160,220,10)):
#     rate_suc_geo_labial = rate_suc_geo_labial_per_freq[rate]
    Z_sim_tarsal = np.array(pd.read_parquet('data/simulation_data_final/tarsal/tarsal_sugar.parquet'))
    Z_sim_geo_tarsal = np.array(pd.read_parquet(f'data/simulation_data_final/tarsal/tarsal_sugar_geosmin_{rate}Hz.parquet'))

    first_vals = np.arange(0, 201, 20)  
    second_vals = np.arange(0, 201, 20)  

    interp_func_tarsal = make_interpolator(Z_sim_tarsal, first_vals)
    interp_func_geo_tarsal = make_interpolator(Z_sim_geo_tarsal, second_vals)


    vals = np.array([10,50,100,500])

    first_only = [(v, 0) for v in vals]
    second_only = [(0, v) for v in vals]
    both = [(v, v) for v in vals]

    c_all_ = np.array((first_only + second_only + both )*2)
    cond_all = np.concatenate([np.zeros(len(first_only+second_only+both)),np.ones(len(first_only+second_only+both))])
    per_all = np.concatenate([experiment_data[modal][cond]['mean'] for cond in ['sugar','sugar_geosmin'] for modal in ['tarsal_atGRN','tarsal_TPN1','tarsal_all']])


    initial_guess = [
        120, 90, 1.2,   # tarsal Hill
        120, 90, 1.2,   # tarsal Hill
        0.1, 13.85, 0.8,   # activation
    ]

    bounds = [
        (0,200), (0,150), (0.1,2.0),     # tarsal
        (0,200), (0,150), (0.1,2.0),     # tarsal
        (0.01,10), (0,95), (0.5,1), 
    ]



    result = minimize(objective_global_wo_constraint,
                      initial_guess,
                      args=(
                            c_all_,
                            cond_all,
                            per_all,
                            interp_func_tarsal,
                            interp_func_geo_tarsal,
                        ),
                      bounds=bounds,
                      method='L-BFGS-B')

    print(rate,result.x)

#     pickle.dump(result,open(f'fitting_result/labial/result_av1a1_{rate}.pkl','wb'))



160 [133.96238105  85.78531183   0.59568464 138.49161674  85.85034578
   0.27768622   0.32025664   5.83559483   0.70124514]
170 [165.34602952  81.20495507   0.66198596 161.34577026  81.84816145
   0.22291861   0.21584663   9.18519273   0.76795368]
180 [122.58749449  89.50990416   0.80557811 128.65128302  87.4942281
   0.28533471   0.40222215   3.88248366   0.6260796 ]
190 [140.92796616  91.57625394   0.83959954 141.61063795  88.08785891
   0.31059785   0.31051857   5.01619794   0.64151512]
200 [121.2121682   89.54881018   0.97709715 120.59101268  89.72363891
   0.39495328   0.43176841   3.11382222   0.59831107]
210 [156.58573196  80.32603974   1.16657533 121.18750962  85.25096208
   0.44468817   0.27194975   4.95546349   0.6157278 ]


# Labellar Fitting 

In [24]:


for rate in list(range(160,220,10)):
#     rate_suc_geo_labial = rate_suc_geo_labial_per_freq[rate]
    Z_sim_labial = np.array(pd.read_parquet('data/simulation_data_final/labellar/labial_sugar.parquet'))
    Z_sim_geo_labial = np.array(pd.read_parquet(f'data/simulation_data_final/labellar/labial_sugar_geosmin_{rate}Hz.parquet'))

    first_vals = np.arange(0, 201, 20)  
    second_vals = np.arange(0, 201, 20)  

    interp_func_labial = make_interpolator(Z_sim_labial, first_vals)
    interp_func_geo_labial = make_interpolator(Z_sim_geo_labial, second_vals)


    vals = np.array([10,50,100,500])

    first_only = [(v, 0) for v in vals]
    second_only = [(0, v) for v in vals]
    both = [(v, v) for v in vals]

    c_all_ = np.array((first_only + second_only + both )*2)
    cond_all = np.concatenate([np.zeros(len(first_only+second_only+both)),np.ones(len(first_only+second_only+both))])
    per_all = np.concatenate([experiment_data[modal][cond]['mean'] for cond in ['sugar','sugar_geosmin'] for modal in ['labial_L1L2','labial_L3','labial_all']])


    initial_guess = [
        60, 90, 1.0,   # labial Hill
        60, 90, 1.0,   # labial Hill
        0.1, 13.85, 0.8,   # activation
    ]

    bounds = [
        (0,100), (0,200), (0,1),     # labial
        (0,100), (0,200), (0,1),     # labial
        (0.01,10), (0,95), (0.5,1), 
    ]


    result = minimize(objective_global_wo_constraint,
                      initial_guess,
                      args=(
                            c_all_,
                            cond_all,
                            per_all,
                            interp_func_labial,
                            interp_func_geo_labial,
                        ),
                      bounds=bounds,
                      method='L-BFGS-B')

    print(rate,result.x)

#     pickle.dump(result,open(f'fitting_result/labial/result_av1a1_{rate}.pkl','wb'))



160 [54.01432051 91.47513769  0.72200749 57.05464728 91.78817853  0.99939936
  0.05940199 20.53512374  0.98965293]
170 [ 49.99039708  92.08216818   0.67833506  47.05469615 100.07102209
   1.           0.08620878  10.76644233   0.8001342 ]
180 [ 51.60478815  92.78824195   0.76777358  50.59006382 102.87125465
   1.           0.08333925  10.63351673   0.78565843]
190 [56.53467615 91.05285787  0.85258413 59.56966931 91.00241771  1.
  0.06271845 16.6043741   0.87558042]
200 [ 62.26260287  94.46482989   0.89469681  76.15162907 111.72135632
   0.87569955   0.05342035  20.99996151   0.92294286]
210 [56.93656321 90.89884847  0.93062466 61.06641343 90.09680863  1.
  0.06486394 14.71658389  0.83210927]


In [20]:
np.set_printoptions(suppress=True)
for rate in range(160,220,10):
    result = pickle.load(open(f'data_wo_constraint/labial/result_av1a1_{rate}.pkl','rb'))
    print(rate,result.x)

160 [54.01432051 91.47513769  0.72200749 57.05464728 91.78817853  0.99939936
  0.05940199 20.53512374  0.98965293]
170 [ 49.99039708  92.08216818   0.67833506  47.05469615 100.07102209
   1.           0.08620878  10.76644233   0.8001342 ]
180 [ 51.60478815  92.78824195   0.76777358  50.59006382 102.87125465
   1.           0.08333925  10.63351673   0.78565843]
190 [56.53467615 91.05285787  0.85258413 59.56966931 91.00241771  1.
  0.06271845 16.6043741   0.87558042]
200 [ 62.26260287  94.46482989   0.89469681  76.15162907 111.72135632
   0.87569955   0.05342035  20.99996151   0.92294286]
210 [56.93656321 90.89884847  0.93062466 61.06641343 90.09680863  1.
  0.06486394 14.71658389  0.83210927]
